# Module 4 Instructor Colab Notebook: Singular Value Decomposition and Low-Rank Approximation

## A Mathematical Bridge to Artificial Intelligence

This module develops singular value decomposition from its geometric and algebraic foundations, then uses it to construct optimal low-rank approximations. The applications show how singular values reveal redundancy, support matrix compression, and separate dominant structure from small-scale noise.

**Recommended pacing:** two or three class meetings. The image-compression and denoising examples can serve as a computational laboratory.


> **How to use this notebook in class**
>
> Begin with the exact two-by-two example before running numerical SVD. Require students to track the shapes of every factor and distinguish full, reduced, and compact forms. In compression examples, discuss approximation error and resource savings together so that compression is never presented as lossless by default.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | SVD factors and singular vectors | Understanding matrix action |
| 2 | Connection with eigenvalues | Constructing singular values and vectors |
| 3 | Geometry and outer products | Ranking independent matrix patterns |
| 4 | Truncated SVD and optimality | Best low-rank approximation |
| 5 | Compression | Reducing linear-layer storage and computation |
| 6 | Redundancy and denoising | Preserving dominant dataset structure |
| Integrated lab | Rank selection | Compressing a document-feature matrix |


## Module learning objectives

By the end of Module 4, students should be able to:

1. define singular values and left and right singular vectors;
2. state and track the shapes in full, reduced, and compact SVD;
3. derive singular values from the eigenvalues of `A` transpose times `A`;
4. construct an exact SVD for a small matrix;
5. interpret SVD as rotation, coordinate-wise scaling, and rotation;
6. express a matrix as a sum of rank-one outer products;
7. define the Frobenius norm and spectral norm of a matrix;
8. construct a truncated SVD and compute its approximation errors;
9. explain the Eckart-Young optimality theorem;
10. evaluate storage, computation, approximation error, energy retention, and numerical rank in AI applications.


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable display and instructor-reveal helpers.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def annotated_heatmap(matrix, title, row_labels=None, column_labels=None, cmap="Blues"):
    matrix = np.asarray(matrix, dtype=float)
    fig, ax = plt.subplots(figsize=(max(5, matrix.shape[1] * 1.2),
                                    max(3.5, matrix.shape[0] * 0.8)))
    image = ax.imshow(matrix, cmap=cmap, aspect="auto")
    if row_labels is not None:
        ax.set_yticks(range(matrix.shape[0]), row_labels)
    if column_labels is not None:
        ax.set_xticks(range(matrix.shape[1]), column_labels)
    ax.set_title(title)
    threshold = (matrix.max() + matrix.min()) / 2 if matrix.size else 0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            color = "white" if matrix[i, j] > threshold else "black"
            ax.text(j, i, f"{matrix[i,j]:g}", ha="center", va="center", color=color)
    fig.colorbar(image, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


> **Notation convention**
>
> Singular values are ordered from largest to smallest. The same ordering must be used for the corresponding columns of `U` and `V`. Software may reverse singular-vector signs; paired sign reversals leave the reconstructed matrix unchanged.

---

# Section 1: What Is Singular Value Decomposition?

## Learning objectives

Students should be able to state the SVD theorem, explain every factor, distinguish factor shapes, and reconstruct a matrix from its singular components.


## 1. Why SVD is needed

Eigendecomposition is powerful but has two limitations:

1. it applies directly only to square matrices; and
2. some square matrices do not have enough independent eigenvectors to be diagonalized.

Singular value decomposition, abbreviated **SVD**, applies to every real rectangular matrix. It provides orthogonal input directions, nonnegative scaling values, and orthogonal output directions.


## 2. Full SVD theorem

For every real matrix

$$
A\in\mathbb{R}^{m\times n},
$$

there exist orthogonal matrices

$$
U\in\mathbb{R}^{m\times m},
\qquad
V\in\mathbb{R}^{n\times n},
$$

and a rectangular diagonal matrix

$$
\Sigma\in\mathbb{R}^{m\times n}
$$

such that

$$
A=U\Sigma V^T.
$$

The diagonal entries of `Sigma` are nonnegative and ordered:

$$
\sigma_1\geq\sigma_2\geq\cdots\geq\sigma_r>0,
$$

followed by zeros. The positive values are the **singular values**, and

$$
r=\operatorname{rank}(A).
$$


## 3. Meaning of the factors

### Right singular vectors

The columns of `V` are **right singular vectors**. They form an orthonormal basis of the input space.

### Singular values

The diagonal entries of `Sigma` are nonnegative scaling factors. A larger singular value identifies a direction that the matrix stretches more strongly.

### Left singular vectors

The columns of `U` are **left singular vectors**. They form an orthonormal basis of the output space.

For every positive singular value,

$$
A\mathbf{v}_i=\sigma_i\mathbf{u}_i.
$$

Thus the matrix takes the input direction `v_i`, scales it by `sigma_i`, and sends it to the output direction `u_i`.


## 4. What does orthogonal mean for a matrix?

A square matrix `Q` is **orthogonal** when

$$
Q^TQ=QQ^T=I.
$$

Therefore,

$$
Q^{-1}=Q^T.
$$

Its columns form an orthonormal basis: each column has L2 norm one, and distinct columns have dot product zero.

Orthogonal transformations preserve dot products and L2 lengths:

$$
\lVert Q\mathbf{x}\rVert_2=\lVert\mathbf{x}\rVert_2.
$$

Geometrically, an orthogonal matrix rotates, reflects, or combines rotation and reflection without stretching.


## 5. Full, reduced, and compact SVD shapes

Let

$$
q=\min(m,n)
$$

and let

$$
r=\operatorname{rank}(A).
$$

| Form | `U` shape | `Sigma` shape | `V` shape | Purpose |
|---|---:|---:|---:|---|
| Full | `m by m` | `m by n` | `n by n` | complete orthonormal bases |
| Reduced | `m by q` | `q by q` | `n by q` | removes structurally unused extra columns |
| Compact | `m by r` | `r by r` | `n by r` | keeps only positive singular-value directions |

In reduced or compact form, the reconstruction is still written

$$
A=U\Sigma V^T,
$$

but the factor shapes follow the selected form.

> **Software note**
>
> NumPy with `full_matrices=False` returns the reduced form. If the matrix is rank deficient, zero or numerically tiny singular directions may still appear; compact form additionally removes those directions.


## 6. Exact SVD example with transparent arithmetic

Let

$$
A=
\begin{bmatrix}
3&0\\
4&0
\end{bmatrix}.
$$

One SVD is

$$
U=
\begin{bmatrix}
\frac{3}{5}&-\frac{4}{5}\\
\frac{4}{5}&\frac{3}{5}
\end{bmatrix},
\qquad
\Sigma=
\begin{bmatrix}
5&0\\
0&0
\end{bmatrix},
\qquad
V=
\begin{bmatrix}
1&0\\
0&1
\end{bmatrix}.
$$

The first right singular vector is

$$
\mathbf{v}_1=
\begin{bmatrix}1\\0\end{bmatrix}.
$$

Apply `A`:

$$
A\mathbf{v}_1
=
\begin{bmatrix}3\\4\end{bmatrix}.
$$

Its length is

$$
\sqrt{3^2+4^2}=5.
$$

Normalize the output:

$$
\mathbf{u}_1
=\frac{1}{5}
\begin{bmatrix}3\\4\end{bmatrix}
=
\begin{bmatrix}\frac{3}{5}\\\frac{4}{5}\end{bmatrix}.
$$

Therefore,

$$
A\mathbf{v}_1=5\mathbf{u}_1.
$$


The second input direction is

$$
\mathbf{v}_2=
\begin{bmatrix}0\\1\end{bmatrix}.
$$

It is sent to zero:

$$
A\mathbf{v}_2
=
\begin{bmatrix}0\\0\end{bmatrix}.
$$

Thus the second singular value is zero. Choose a unit output direction perpendicular to `u1`:

$$
\mathbf{u}_2
=
\begin{bmatrix}-\frac{4}{5}\\\frac{3}{5}\end{bmatrix}.
$$

Now reconstruct. First,

$$
U\Sigma
=
\begin{bmatrix}
\frac{3}{5}&-\frac{4}{5}\\
\frac{4}{5}&\frac{3}{5}
\end{bmatrix}
\begin{bmatrix}
5&0\\0&0
\end{bmatrix}
=
\begin{bmatrix}
3&0\\4&0
\end{bmatrix}.
$$

Since `V` is the identity,

$$
U\Sigma V^T=A.
$$

There is one positive singular value, so

$$
\operatorname{rank}(A)=1.
$$


In [ ]:
A_svd_exact = np.array([[3.0, 0.0], [4.0, 0.0]])
U_exact, s_exact, Vt_exact = np.linalg.svd(A_svd_exact, full_matrices=True)

print("A =\n", A_svd_exact)
print("U =\n", U_exact)
print("singular values:", s_exact)
print("V transpose =\n", Vt_exact)
print("reconstruction =\n", U_exact @ np.diag(s_exact) @ Vt_exact)
print("rank from positive singular values:", np.count_nonzero(s_exact > 1e-12))


> **Sign ambiguity**
>
> NumPy may return the negatives of singular vectors shown by hand. If both members of a paired outer product change sign, then
>
> $$
> (-\mathbf{u}_i)\sigma_i(-\mathbf{v}_i)^T
> =\mathbf{u}_i\sigma_i\mathbf{v}_i^T.
> $$
>
> The decomposition represents the same matrix.


> **Common mistakes in Section 1**
>
> - Assuming SVD applies only to square matrices.
> - Writing `V` instead of `V` transpose in the reconstruction.
> - Allowing negative singular values.
> - Mixing full, reduced, and compact shapes.
> - Assuming singular vectors are unique when singular values repeat.
> - Treating paired sign changes as different decompositions in substance.

## Section 1 terminology

| Term | Meaning |
|---|---|
| SVD | Factorization `A=U Sigma V` transpose |
| Singular value | Nonnegative scaling magnitude |
| Right singular vector | Orthonormal input direction |
| Left singular vector | Orthonormal output direction |
| Orthogonal matrix | Matrix whose transpose equals its inverse |
| Full SVD | Includes complete input and output bases |
| Reduced SVD | Uses the smaller dimension `q` |
| Compact SVD | Keeps only positive singular-value directions |


## Section 1 worksheet

1. State the full SVD shapes for a `5 by 3` matrix.
2. State the reduced SVD shapes.
3. If the matrix rank is `2`, state the compact SVD shapes.
4. Explain the meaning of

$$
A\mathbf{v}_i=\sigma_i\mathbf{u}_i.
$$

5. Why are singular values never negative?


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. Full form: `U` is `5 by 5`, `Sigma` is `5 by 3`, and `V` is `3 by 3`.

2. Here

$$
q=\min(5,3)=3.
$$

Reduced form: `U` is `5 by 3`, `Sigma` is `3 by 3`, and `V` is `3 by 3`.

3. Compact form with rank `2`: `U` is `5 by 2`, `Sigma` is `2 by 2`, and `V` is `3 by 2`.

4. The matrix maps the right singular input direction `v_i` to the left singular output direction `u_i`, with length multiplied by `sigma_i`.

5. Singular values measure stretching magnitude and are defined as square roots of nonnegative eigenvalues of `A` transpose times `A`.
""")


---

# Section 2: Constructing SVD from Eigenvalues

## Learning objectives

Students should be able to derive the connection with `A` transpose times `A`, prove nonnegativity, construct singular vectors, and explain why SVD exists for every real matrix.


## 1. Right singular vectors from `A` transpose times `A`

Suppose

$$
A\mathbf{v}_i=\sigma_i\mathbf{u}_i.
$$

For a positive singular value, the corresponding reverse relation is

$$
A^T\mathbf{u}_i=\sigma_i\mathbf{v}_i.
$$

Apply `A` transpose to the first relation:

$$
A^TA\mathbf{v}_i
=A^T(\sigma_i\mathbf{u}_i).
$$

Use linearity:

$$
=\sigma_iA^T\mathbf{u}_i.
$$

Use the reverse relation:

$$
=\sigma_i(\sigma_i\mathbf{v}_i)
=\sigma_i^2\mathbf{v}_i.
$$

Therefore, right singular vectors are eigenvectors of

$$
A^TA,
$$

and the associated eigenvalues are squared singular values.


## 2. Why the eigenvalues are nonnegative

The matrix

$$
A^TA
$$

is symmetric. The scalar expression

$$
\mathbf{x}^TM\mathbf{x}
$$

is called the **quadratic form** associated with a square matrix `M`. The matrix `A` transpose times `A` is **positive semidefinite**, meaning its quadratic form is nonnegative:

$$
\mathbf{x}^TA^TA\mathbf{x}\geq 0
$$

for every vector `x`.

Indeed,

$$
\mathbf{x}^TA^TA\mathbf{x}
=(A\mathbf{x})^T(A\mathbf{x})
=\lVert A\mathbf{x}\rVert_2^2
\geq 0.
$$

If `v` is a unit eigenvector with eigenvalue `lambda`, then

$$
\lambda
=\lambda\mathbf{v}^T\mathbf{v}
=\mathbf{v}^T(A^TA)\mathbf{v}
\geq 0.
$$

Thus,

$$
\sigma_i=\sqrt{\lambda_i}
$$

is real and nonnegative.


## 3. Constructing left singular vectors

After finding a unit right singular vector and a positive singular value, define

$$
\mathbf{u}_i
=\frac{A\mathbf{v}_i}{\sigma_i}.
$$

This vector has unit length:

$$
\lVert\mathbf{u}_i\rVert_2^2
=\frac{(A\mathbf{v}_i)^T(A\mathbf{v}_i)}{\sigma_i^2}.
$$

$$
=\frac{\mathbf{v}_i^TA^TA\mathbf{v}_i}{\sigma_i^2}.
$$

Since

$$
A^TA\mathbf{v}_i=\sigma_i^2\mathbf{v}_i
$$

and `v_i` has unit length,

$$
\lVert\mathbf{u}_i\rVert_2^2
=\frac{\sigma_i^2}{\sigma_i^2}=1.
$$

Distinct positive-singular-value directions can be chosen orthogonal, producing orthonormal left singular vectors.


## 4. Detailed construction for the exact example

For

$$
A=
\begin{bmatrix}3&0\\4&0\end{bmatrix},
$$

compute

$$
A^TA
=
\begin{bmatrix}3&4\\0&0\end{bmatrix}
\begin{bmatrix}3&0\\4&0\end{bmatrix}
=
\begin{bmatrix}25&0\\0&0\end{bmatrix}.
$$

Its eigenvalues are `25` and `0`, so the singular values are

$$
\sigma_1=\sqrt{25}=5,
\qquad
\sigma_2=\sqrt{0}=0.
$$

Associated unit eigenvectors are

$$
\mathbf{v}_1=
\begin{bmatrix}1\\0\end{bmatrix},
\qquad
\mathbf{v}_2=
\begin{bmatrix}0\\1\end{bmatrix}.
$$

For the positive singular value,

$$
\mathbf{u}_1
=\frac{A\mathbf{v}_1}{5}
=\frac{1}{5}
\begin{bmatrix}3\\4\end{bmatrix}
=
\begin{bmatrix}\frac35\\\frac45\end{bmatrix}.
$$

Complete the output basis using a perpendicular unit vector.


In [ ]:
AtA_exact = A_svd_exact.T @ A_svd_exact
lambda_exact, V_exact_eig = np.linalg.eigh(AtA_exact)
order_exact = np.argsort(lambda_exact)[::-1]
lambda_exact = lambda_exact[order_exact]
V_exact_eig = V_exact_eig[:, order_exact]
sigma_from_eigen = np.sqrt(np.maximum(lambda_exact, 0))

print("A^T A =\n", AtA_exact)
print("eigenvalues:", lambda_exact)
print("singular values from square roots:", sigma_from_eigen)
print("right singular directions:\n", V_exact_eig)


## 5. Why SVD exists: proof outline

1. The matrix

$$
A^TA
$$

is real and symmetric.

2. By the spectral theorem, it has an orthonormal eigenbasis of right singular vectors:

$$
\mathbf{v}_1,\ldots,\mathbf{v}_n.
$$

3. Its eigenvalues are nonnegative, so define

$$
\sigma_i=\sqrt{\lambda_i}.
$$

4. For every positive singular value, define

$$
\mathbf{u}_i=\frac{A\mathbf{v}_i}{\sigma_i}.
$$

The preceding calculation proves these vectors are orthonormal.

5. Extend the positive-singular-value left vectors to an orthonormal basis of the output space if necessary.

6. Placing the bases into `U` and `V`, and the singular values into `Sigma`, produces

$$
A=U\Sigma V^T.
$$

This construction does not require `A` to be square or diagonalizable.


## 6. SVD versus eigendecomposition

| Feature | Eigendecomposition | SVD |
|---|---|---|
| Matrix shape | square | any rectangular shape |
| Existence | not every square matrix is diagonalizable | every real matrix has an SVD |
| Scaling values | eigenvalues may be negative or complex | singular values are real and nonnegative |
| Directions | eigenvectors may not be orthogonal | left and right singular vectors are orthonormal |
| Formula | `A=PDP` inverse | `A=U Sigma V` transpose |
| Main geometric role | invariant directions in one space | input-output stretching directions |

For a real symmetric positive semidefinite matrix, the two decompositions align closely: left and right singular vectors can equal the eigenvectors, and singular values equal the nonnegative eigenvalues.


> **Common mistakes in Section 2**
>
> - Using eigenvalues of `A` itself for a general rectangular matrix.
> - Forgetting to take square roots of eigenvalues of `A` transpose times `A`.
> - Dividing by a zero singular value when constructing a left singular vector.
> - Assuming `A` transpose times `A` and `A A` transpose have the same shape.
> - Confusing positive semidefinite with every entry being nonnegative.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Positive semidefinite | Quadratic form is nonnegative for every vector |
| Quadratic form | Scalar expression `x` transpose times `M` times `x` |
| Spectral theorem | Real symmetric matrix has an orthonormal eigenbasis |
| Squared singular value | Eigenvalue of `A` transpose times `A` |
| Null direction | Right singular direction associated with singular value zero |
| Input space | Space containing right singular vectors |
| Output space | Space containing left singular vectors |


## Section 2 worksheet

Let

$$
A=
\begin{bmatrix}
2&0\\
0&1\\
0&0
\end{bmatrix}.
$$

1. Compute `A` transpose times `A`.
2. Find its eigenvalues and eigenvectors.
3. Find the singular values.
4. Construct the two reduced left singular vectors.
5. State the reduced SVD factor shapes.


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
### 1. Product

$$
A^TA=
\begin{bmatrix}4&0\\0&1\end{bmatrix}.
$$

### 2. Eigenpairs

The eigenvalues are `4` and `1`, with coordinate eigenvectors

$$
\mathbf{v}_1=\begin{bmatrix}1\\0\end{bmatrix},
\qquad
\mathbf{v}_2=\begin{bmatrix}0\\1\end{bmatrix}.
$$

### 3. Singular values

$$
\sigma_1=\sqrt4=2,
\qquad
\sigma_2=\sqrt1=1.
$$

### 4. Left singular vectors

$$
\mathbf{u}_1=\frac{A\mathbf{v}_1}{2}
=\begin{bmatrix}1\\0\\0\end{bmatrix},
$$

$$
\mathbf{u}_2=\frac{A\mathbf{v}_2}{1}
=\begin{bmatrix}0\\1\\0\end{bmatrix}.
$$

### 5. Reduced shapes

The matrix is `3 by 2`, so `q=2`. Reduced `U` is `3 by 2`, `Sigma` is `2 by 2`, and `V` is `2 by 2`.
""")


---

# Section 3: Geometric and Rank-One Interpretations

## Learning objectives

Students should be able to interpret SVD as a sequence of orthogonal and scaling transformations, expand a matrix into rank-one terms, and connect singular values with matrix rank and captured energy.


## 1. Rotation-scaling-rotation interpretation

Apply

$$
A=U\Sigma V^T
$$

to a vector:

$$
A\mathbf{x}=U\Sigma V^T\mathbf{x}.
$$

Read from right to left:

1. `V` transpose expresses the input in right-singular-vector coordinates;
2. `Sigma` scales each coordinate by its singular value; and
3. `U` moves the scaled coordinates into left-singular-vector directions.

Orthogonal factors preserve lengths. All stretching and collapse occur in `Sigma`.

The unit circle is therefore mapped to an ellipse. Its principal semiaxis lengths are the singular values, and its axis directions are the left singular vectors.


In [ ]:
A_geometry = np.array([[3.0, 1.0], [1.0, 2.0]])
U_geometry, s_geometry, Vt_geometry = np.linalg.svd(A_geometry)
theta_geometry = np.linspace(0, 2*np.pi, 500)
circle_geometry = np.vstack([np.cos(theta_geometry), np.sin(theta_geometry)])
ellipse_geometry = A_geometry @ circle_geometry

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].plot(circle_geometry[0], circle_geometry[1], linewidth=3)
axes[0].set_title("Input unit circle")
axes[1].plot(ellipse_geometry[0], ellipse_geometry[1], linewidth=3, color="#e45756")
for i in range(2):
    axis_vector = s_geometry[i] * U_geometry[:, i]
    axes[1].plot([-axis_vector[0], axis_vector[0]], [-axis_vector[1], axis_vector[1]],
                 linewidth=2.5, label=f"axis length {s_geometry[i]:.3f}")
axes[1].set_title("Output ellipse under A")
axes[1].legend()
for ax in axes:
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_aspect("equal")
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-4.5, 4.5)
plt.tight_layout()
plt.show()

print("Singular values:", s_geometry)
print("Right singular vectors as columns:\n", Vt_geometry.T)
print("Left singular vectors as columns:\n", U_geometry)


## 2. Outer products

For column vectors

$$
\mathbf{u}\in\mathbb{R}^{m},
\qquad
\mathbf{v}\in\mathbb{R}^{n},
$$

the **outer product** is

$$
\mathbf{u}\mathbf{v}^T
\in\mathbb{R}^{m\times n}.
$$

Its entry in row `i`, column `j` is

$$
u_iv_j.
$$

Every column is a scalar multiple of `u`, so a nonzero outer product has rank one.


### Detailed outer-product example

Let

$$
\mathbf{u}=
\begin{bmatrix}1\\2\\-1\end{bmatrix},
\qquad
\mathbf{v}=
\begin{bmatrix}3\\4\end{bmatrix}.
$$

Then

$$
\mathbf{u}\mathbf{v}^T
=
\begin{bmatrix}1\\2\\-1\end{bmatrix}
\begin{bmatrix}3&4\end{bmatrix}.
$$

Multiply each entry of `u` by the entire row `v` transpose:

$$
=
\begin{bmatrix}
1(3)&1(4)\\
2(3)&2(4)\\
(-1)(3)&(-1)(4)
\end{bmatrix}
=
\begin{bmatrix}
3&4\\
6&8\\
-3&-4
\end{bmatrix}.
$$

The second column is `4/3` times the first, confirming rank one.


In [ ]:
u_outer = np.array([1.0, 2.0, -1.0])
v_outer = np.array([3.0, 4.0])
outer_example = np.outer(u_outer, v_outer)
print("Outer product =\n", outer_example)
print("Rank:", np.linalg.matrix_rank(outer_example))


## 3. SVD as a sum of ranked patterns

The compact SVD expands as

$$
A
=\sigma_1\mathbf{u}_1\mathbf{v}_1^T
+\sigma_2\mathbf{u}_2\mathbf{v}_2^T
+\cdots
+\sigma_r\mathbf{u}_r\mathbf{v}_r^T.
$$

Each term is rank one. The singular value weights the importance of that pattern.

This expansion explains rank directly: there are exactly `r` positive singular-value patterns.

It also motivates truncation: keep the first few strongest terms and discard smaller ones.


In [ ]:
A_components = np.array([
    [4.0, 3.0, 2.0],
    [3.0, 2.0, 1.0],
    [2.0, 1.0, 3.0],
    [1.0, 0.0, 2.0],
])
U_comp, s_comp, Vt_comp = np.linalg.svd(A_components, full_matrices=False)
rank_one_terms = [s_comp[i] * np.outer(U_comp[:,i], Vt_comp[i,:]) for i in range(len(s_comp))]

print("Singular values:", s_comp)
print("Sum of all rank-one terms agrees:", np.allclose(sum(rank_one_terms), A_components))

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
matrices = [A_components, *rank_one_terms]
titles = ["Original A", *[f"term {i+1}\nσ={s_comp[i]:.2f}" for i in range(3)]]
for ax, matrix, title in zip(axes, matrices, titles):
    ax.imshow(matrix, cmap="coolwarm", aspect="auto")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("SVD decomposes a matrix into ranked patterns", y=1.02)
plt.tight_layout()
plt.show()


## 4. Singular-value energy

The squared Frobenius norm of a matrix equals the sum of squared singular values:

$$
\lVert A\rVert_F^2
=\sum_{i,j}a_{ij}^2
=\sum_{i=1}^{r}\sigma_i^2.
$$

The fraction of singular-value energy retained by the first `k` components is

$$
\frac{\sum_{i=1}^{k}\sigma_i^2}
{\sum_{i=1}^{r}\sigma_i^2}.
$$

This is often used to guide rank selection, but it is not identical to downstream prediction accuracy. A small-energy direction can still matter for a particular task.


In [ ]:
energy_comp = s_comp**2
cumulative_energy_comp = np.cumsum(energy_comp) / energy_comp.sum()
for k, fraction in enumerate(cumulative_energy_comp, start=1):
    print(f"first {k} component(s): {100*fraction:.2f}% energy")


> **Common mistakes in Section 3**
>
> - Confusing an outer product with a dot product.
> - Forgetting that matrix transformations are read from right to left.
> - Calling each rank-one term an eigenvector.
> - Adding singular values rather than squared singular values when computing Frobenius energy.
> - Assuming the largest-energy directions must be the most predictive directions.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Outer product | Matrix formed from all pairwise coordinate products of two vectors |
| Rank-one term | Matrix containing one independent row and column pattern |
| Principal axis | Ellipse direction associated with a singular vector |
| Singular-value energy | Squared singular value contribution to squared Frobenius norm |
| Cumulative energy | Fraction of total squared singular values retained |


## Section 3 guided worksheet

1. Compute the outer product for

   $$
   u=\begin{bmatrix}1\\2\end{bmatrix},
   \qquad
   v=\begin{bmatrix}3\\-1\end{bmatrix}.
   $$

   Explain why the resulting nonzero matrix has rank one.

2. A matrix has singular values $$6,2,1.$$ Compute the fraction of singular-value energy retained by the first component and by the first two components.

3. In the geometric interpretation of

   $$
   A=U\Sigma V^T,
   $$

   describe the role of $$V^T,$$ $$\Sigma,$$ and $$U$$ in that order.


In [ ]:
reveal_markdown("Section 3 worksheet solution", r"""
1. An outer product multiplies every coordinate of $$u$$ by every coordinate of $$v$$:

   $$
   uv^T
   =
   \begin{bmatrix}1\\2\end{bmatrix}
   \begin{bmatrix}3&-1\end{bmatrix}
   =
   \begin{bmatrix}
   1(3)&1(-1)\\
   2(3)&2(-1)
   \end{bmatrix}
   =
   \begin{bmatrix}
   3&-1\\
   6&-2
   \end{bmatrix}.
   $$

   Its second row is twice its first row, and each column is a multiple of $$u.$$ Thus it has only one independent row and one independent column, so its rank is one.

2. Total singular-value energy is

   $$
   6^2+2^2+1^2=36+4+1=41.
   $$

   The first component retains

   $$
   \frac{36}{41}\approx 0.8780=87.80\%.
   $$

   The first two retain

   $$
   \frac{36+4}{41}=\frac{40}{41}\approx 0.9756=97.56\%.
   $$

3. Reading the product from right to left, $$V^T$$ rotates or reflects the input into right-singular-vector coordinates, $$\Sigma$$ stretches those coordinates by the singular values, and $$U$$ rotates or reflects the result into the output space.
""")


---

# Section 4: Truncated SVD and Optimal Low-Rank Approximation

## Learning objectives

Students should be able to form a truncated SVD, compute matrix approximation errors, state and explain the Eckart-Young theorem, and select rank using error or energy criteria.


## 1. Truncated SVD

Order singular values from largest to smallest. The rank-`k` truncated SVD is

$$
A_k
=\sum_{i=1}^{k}\sigma_i\mathbf{u}_i\mathbf{v}_i^T
=U_k\Sigma_kV_k^T.
$$

It keeps the first `k` singular directions and discards the rest.

The approximation has rank at most `k`:

$$
\operatorname{rank}(A_k)\leq k.
$$

If

$$
k<\operatorname{rank}(A),
$$

information is discarded, so the approximation is generally not exact.


## 2. Two matrix error norms

### Frobenius norm

The **Frobenius norm** treats all entries as one long vector:

$$
\lVert A\rVert_F
=\sqrt{\sum_{i,j}a_{ij}^2}.
$$

For truncated SVD error,

$$
\lVert A-A_k\rVert_F
=\sqrt{\sum_{i=k+1}^{r}\sigma_i^2}.
$$

### Spectral norm

The **spectral norm** is maximum L2 stretching:

$$
\lVert A\rVert_2
=\max_{\lVert\mathbf{x}\rVert_2=1}
\lVert A\mathbf{x}\rVert_2
=\sigma_1.
$$

For truncated SVD error,

$$
\lVert A-A_k\rVert_2=\sigma_{k+1}.
$$


## 3. Exact two-by-two truncation example

Let

$$
A=
\begin{bmatrix}
3&1\\
1&3
\end{bmatrix}.
$$

This symmetric positive matrix has orthonormal singular vectors

$$
\mathbf{u}_1=\mathbf{v}_1
=\frac{1}{\sqrt2}
\begin{bmatrix}1\\1\end{bmatrix},
$$

$$
\mathbf{u}_2=\mathbf{v}_2
=\frac{1}{\sqrt2}
\begin{bmatrix}1\\-1\end{bmatrix},
$$

with singular values

$$
\sigma_1=4,
\qquad
\sigma_2=2.
$$

The best rank-one approximation is

$$
A_1=4\mathbf{u}_1\mathbf{v}_1^T.
$$

Compute the outer product:

$$
\mathbf{u}_1\mathbf{v}_1^T
=\frac12
\begin{bmatrix}
1&1\\1&1
\end{bmatrix}.
$$

Therefore,

$$
A_1
=4\left(\frac12
\begin{bmatrix}1&1\\1&1\end{bmatrix}\right)
=
\begin{bmatrix}2&2\\2&2\end{bmatrix}.
$$


The error matrix is

$$
A-A_1
=
\begin{bmatrix}3&1\\1&3\end{bmatrix}
-
\begin{bmatrix}2&2\\2&2\end{bmatrix}
=
\begin{bmatrix}1&-1\\-1&1\end{bmatrix}.
$$

The spectral error equals the discarded singular value:

$$
\lVert A-A_1\rVert_2=\sigma_2=2.
$$

The Frobenius error is

$$
\lVert A-A_1\rVert_F
=\sqrt{1^2+(-1)^2+(-1)^2+1^2}
=\sqrt4
=2.
$$

Because only one singular value was discarded, the singular-value formula also gives

$$
\sqrt{\sigma_2^2}=2.
$$


In [ ]:
A_trunc_exact = np.array([[3.0, 1.0], [1.0, 3.0]])
U_trunc, s_trunc, Vt_trunc = np.linalg.svd(A_trunc_exact)
A1_trunc = s_trunc[0] * np.outer(U_trunc[:,0], Vt_trunc[0,:])
error_trunc = A_trunc_exact - A1_trunc

print("singular values:", s_trunc)
print("rank-one approximation:\n", A1_trunc)
print("error:\n", error_trunc)
print("Frobenius error:", np.linalg.norm(error_trunc, "fro"))
print("spectral error:", np.linalg.norm(error_trunc, 2))


## 4. Eckart-Young theorem

The **Eckart-Young theorem** states that truncated SVD gives the best rank-at-most-`k` approximation in both the spectral and Frobenius norms.

For every matrix `B` with

$$
\operatorname{rank}(B)\leq k,
$$

$$
\lVert A-A_k\rVert_2
\leq
\lVert A-B\rVert_2,
$$

and

$$
\lVert A-A_k\rVert_F
\leq
\lVert A-B\rVert_F.
$$

The optimal errors are

$$
\lVert A-A_k\rVert_2=\sigma_{k+1},
$$

$$
\lVert A-A_k\rVert_F
=\sqrt{\sum_{i=k+1}^{r}\sigma_i^2}.
$$


## 5. Why truncated SVD is optimal in spectral norm

Let `B` have rank at most `k`. Consider the `k+1` dimensional subspace

$$
S=\operatorname{span}
\{\mathbf{v}_1,\ldots,\mathbf{v}_{k+1}\}.
$$

The restriction of `B` to this subspace cannot be one-to-one because its output rank is at most `k`. Therefore, there exists a unit vector

$$
\mathbf{x}\in S
$$

such that

$$
B\mathbf{x}=\mathbf{0}.
$$

Then

$$
\lVert A-B\rVert_2
\geq\lVert(A-B)\mathbf{x}\rVert_2.
$$

Since `B x` is zero,

$$
=\lVert A\mathbf{x}\rVert_2.
$$

Write

$$
\mathbf{x}=c_1\mathbf{v}_1+\cdots+c_{k+1}\mathbf{v}_{k+1},
$$

with

$$
\sum_{i=1}^{k+1}c_i^2=1.
$$

Then

$$
\lVert A\mathbf{x}\rVert_2^2
=\sum_{i=1}^{k+1}\sigma_i^2c_i^2
\geq\sigma_{k+1}^2\sum_{i=1}^{k+1}c_i^2
=\sigma_{k+1}^2.
$$

Therefore,

$$
\lVert A-B\rVert_2\geq\sigma_{k+1}.
$$

Truncated SVD attains exactly this error, so no rank-`k` matrix can do better in spectral norm.


## 6. Selecting a truncation rank

Common strategies include:

1. choose the smallest `k` reaching a target cumulative energy;
2. inspect a **scree plot** for a gap or elbow in singular values;
3. choose `k` using validation performance on the downstream task;
4. choose `k` to meet a memory or latency budget; and
5. balance relative error against compression ratio.

No single threshold is correct for every dataset or AI model.


In [ ]:
# Interactive reconstruction of the component example.
def truncated_reconstruction_demo(k=1):
    k = int(k)
    Ak = (U_comp[:, :k] * s_comp[:k]) @ Vt_comp[:k, :]
    fro_error = np.linalg.norm(A_components - Ak, "fro")
    retained = np.sum(s_comp[:k]**2) / np.sum(s_comp**2)
    print(f"rank k = {k}")
    print(f"Frobenius error = {fro_error:.6f}")
    print(f"energy retained = {100*retained:.2f}%")
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(A_components, cmap="coolwarm", aspect="auto")
    axes[0].set_title("Original")
    axes[1].imshow(Ak, cmap="coolwarm", aspect="auto")
    axes[1].set_title(f"Rank-{k} approximation")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

if WIDGETS_AVAILABLE:
    interact(truncated_reconstruction_demo,
             k=IntSlider(value=1, min=1, max=len(s_comp), step=1, description="rank"))
else:
    truncated_reconstruction_demo(2)


> **Common mistakes in Section 4**
>
> - Keeping the smallest singular values instead of the largest.
> - Truncating `U`, `Sigma`, and `V` inconsistently.
> - Claiming low-rank approximation is exact when discarded singular values are positive.
> - Confusing Frobenius and spectral errors.
> - Using energy retention as a guaranteed accuracy measure.
> - Assuming an arbitrary rank-`k` factorization is as good as truncated SVD.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Truncated SVD | Sum of the first `k` singular components |
| Frobenius norm | Square root of the sum of squared entries |
| Spectral norm | Maximum L2 stretching, equal to largest singular value |
| Eckart-Young theorem | Optimality of truncated SVD among low-rank approximations |
| Scree plot | Plot of singular values by index |
| Relative error | Error divided by original matrix norm |
| Rank-selection criterion | Rule used to choose retained component count |


## Section 4 worksheet

A matrix has singular values

$$
10,\quad 4,\quad 1.
$$

1. Find the rank.
2. Find the spectral error of the best rank-one approximation.
3. Find its Frobenius error.
4. Compute the energy proportion retained by rank one.
5. Find the spectral and Frobenius errors of the best rank-two approximation.


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. All three singular values are positive, so the rank is `3`.

2. Rank one discards singular values `4` and `1`. The spectral error is the largest discarded value:

$$
4.
$$

3. The Frobenius error is

$$
\sqrt{4^2+1^2}=\sqrt{17}.
$$

4. The retained energy fraction is

$$
\frac{10^2}{10^2+4^2+1^2}
=\frac{100}{117}
\approx 85.47\%.
$$

5. Rank two discards only `1`, so both spectral and Frobenius errors equal `1`.
""")


---

# Section 5: AI Application—Compressing a Weight Matrix

## Learning objectives

Students should be able to count parameters and operations in dense and factorized layers, construct SVD factors, measure output error, and explain the tradeoff between compression rank and model fidelity.


## 1. Factorizing a linear layer

Suppose a layer computes

$$
\mathbf{y}=W\mathbf{x},
$$

with

$$
W\in\mathbb{R}^{m\times n}.
$$

Using rank-`k` truncated SVD,

$$
W\approx U_k\Sigma_kV_k^T.
$$

Absorb the singular values into one factor:

$$
L=U_k\Sigma_k
\in\mathbb{R}^{m\times k},
$$

$$
R=V_k^T
\in\mathbb{R}^{k\times n}.
$$

Then

$$
W\mathbf{x}\approx L(R\mathbf{x}).
$$

The original layer is replaced by two thinner linear transformations.


## 2. Parameter and multiplication counts

The dense matrix stores

$$
mn
$$

parameters and requires approximately `mn` multiply-add contributions per input vector.

The two factors store

$$
mk+kn=k(m+n)
$$

parameters and require approximately the same factorized count per input.

Compression is beneficial only if

$$
k(m+n)<mn.
$$


### Large-layer example

Let

$$
W\in\mathbb{R}^{768\times 3072}
$$

and choose

$$
k=64.
$$

The dense parameter count is

$$
768(3072)=2{,}359{,}296.
$$

The factor count is

$$
64(768+3072)=64(3840)=245{,}760.
$$

The retained fraction is

$$
\frac{245{,}760}{2{,}359{,}296}
\approx 0.1042.
$$

Thus the factors use about `10.42` percent of the dense parameters, a reduction of about `89.58` percent.

This arithmetic describes storage potential, not guaranteed accuracy or wall-clock speed. Hardware, batching, memory access, and software kernels also matter.


In [ ]:
m_weight, n_weight, k_weight = 768, 3072, 64
dense_weight_count = m_weight*n_weight
factor_weight_count = k_weight*(m_weight+n_weight)
print("Dense parameter count:", dense_weight_count)
print("Factor parameter count:", factor_weight_count)
print("Retained percentage:", 100*factor_weight_count/dense_weight_count)
print("Reduction percentage:", 100*(1-factor_weight_count/dense_weight_count))


## 3. Small controlled compression experiment

To make rank effects visible, construct a weight matrix whose singular values decay. Then compare exact and compressed layer outputs for a batch of inputs.

The relative weight error is

$$
\frac{\lVert W-W_k\rVert_F}{\lVert W\rVert_F}.
$$

The relative output error is

$$
\frac{\lVert XW^T-XW_k^T\rVert_F}
{\lVert XW^T\rVert_F}.
$$

The output error depends on both the matrix approximation and which input directions appear in `X`.


In [ ]:
rng4 = np.random.default_rng(42)
Q_left, _ = np.linalg.qr(rng4.normal(size=(20, 12)))
Q_right, _ = np.linalg.qr(rng4.normal(size=(12, 12)))
designed_singular_values = np.array([10, 7, 5, 3, 2, 1, 0.7, 0.4, 0.2, 0.1, 0.05, 0.02])
W_controlled = Q_left @ np.diag(designed_singular_values) @ Q_right.T
X_controlled = rng4.normal(size=(50, 12))
Y_exact_controlled = X_controlled @ W_controlled.T

U_w, s_w, Vt_w = np.linalg.svd(W_controlled, full_matrices=False)
ranks_w = np.arange(1, len(s_w)+1)
weight_errors_w = []
output_errors_w = []

for k in ranks_w:
    Wk = (U_w[:, :k] * s_w[:k]) @ Vt_w[:k, :]
    Yk = X_controlled @ Wk.T
    weight_errors_w.append(np.linalg.norm(W_controlled-Wk, "fro") / np.linalg.norm(W_controlled, "fro"))
    output_errors_w.append(np.linalg.norm(Y_exact_controlled-Yk, "fro") / np.linalg.norm(Y_exact_controlled, "fro"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].semilogy(ranks_w, s_w, marker="o")
axes[0].set_xlabel("singular-value index")
axes[0].set_ylabel("singular value")
axes[0].set_title("Designed singular-value decay")
axes[1].plot(ranks_w, weight_errors_w, marker="o", label="relative weight error")
axes[1].plot(ranks_w, output_errors_w, marker="s", label="relative output error")
axes[1].set_xlabel("retained rank")
axes[1].set_ylabel("relative Frobenius error")
axes[1].set_title("Compression error decreases with retained rank")
axes[1].legend()
plt.tight_layout()
plt.show()


## 4. Practical compression workflow

1. Obtain a trained weight matrix.
2. Compute or approximate its leading singular triplets.
3. Choose candidate ranks based on resources and singular-value decay.
4. Replace the dense layer by two factors.
5. Measure validation accuracy, latency, memory, and numerical behavior.
6. Optionally fine-tune the factorized model.

> **Important limitation**
>
> The Eckart-Young theorem minimizes matrix norm error. It does not directly minimize the neural network's task loss. Fine-tuning and downstream evaluation remain necessary.


> **Common mistakes in Section 5**
>
> - Counting only one factor's parameters.
> - Applying factors in the wrong order.
> - Assuming parameter reduction guarantees equal runtime reduction.
> - Choosing rank only from energy without testing the task.
> - Forgetting that biases and other layer costs remain.
> - Claiming SVD compression is lossless when positive singular values are discarded.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Dense layer | Linear layer storing a full weight matrix |
| Factorized layer | Sequence of thin linear transformations |
| Parameter count | Number of stored trainable numerical values |
| Latency | Time required to produce an output |
| Fine-tuning | Additional training after modifying a model |
| Downstream evaluation | Measuring performance on the actual intended task |


## Section 5 guided worksheet

1. A weight matrix has shape $$400\times 600$$ and is approximated with rank $$20.$$ Compute the dense parameter count and the two-factor parameter count. What percentages are retained and removed?

2. Write suitable factor shapes

   $$
   W\approx LR
   $$

   for this matrix. If $$x\in\mathbb{R}^{600},$$ which factor acts first, and what is the dimension of the intermediate vector?

3. Explain why the Eckart-Young theorem does not guarantee unchanged model accuracy after a neural-network layer is compressed.

4. Name practical quantities that should be measured after compression.


In [ ]:
reveal_markdown("Section 5 worksheet solution", r"""
1. The dense matrix stores

   $$
   400(600)=240{,}000
   $$

   parameters. Rank-$$20$$ factors store

   $$
   400(20)+20(600)=8{,}000+12{,}000=20{,}000.
   $$

   Therefore the retained percentage is

   $$
   \frac{20{,}000}{240{,}000}(100\%)
   =8.33\%,
   $$

   and the reduction is approximately

   $$
   100\%-8.33\%=91.67\%.
   $$

2. Choose

   $$
   L\in\mathbb{R}^{400\times 20},
   \qquad
   R\in\mathbb{R}^{20\times 600}.
   $$

   Because matrix products act from right to left, first compute

   $$
   Rx\in\mathbb{R}^{20},
   $$

   and then compute

   $$
   L(Rx)\in\mathbb{R}^{400}.
   $$

3. Eckart-Young minimizes a matrix norm such as $$\lVert W-W_k\rVert_F.$$ A network's task loss also depends on its inputs, later layers, nonlinearities, rare cases, and decision boundaries. The smallest weight-matrix error is not necessarily the smallest prediction error.

4. Relevant measurements include validation accuracy or task loss, parameter memory, end-to-end latency, throughput, energy use, numerical stability, and performance on important subgroups or rare examples. Fine-tuned and non-fine-tuned versions should be compared when appropriate.
""")


---

# Section 6: Dataset Redundancy, Numerical Rank, and Denoising

## Learning objectives

Students should be able to interpret singular-value decay, distinguish exact from numerical rank, reconstruct dominant structure, and explain why truncation can reduce noise but may also remove weak useful signals.


## 1. Exact rank versus numerical rank

In exact mathematics, rank is the number of strictly positive singular values.

With floating-point or measured data, noise can make every singular value technically positive. A **numerical rank** uses a tolerance:

$$
\operatorname{rank}_{\tau}(A)
=\#\{i:\sigma_i>\tau\}.
$$

A common scale-aware tolerance has the form

$$
\tau
=\max(m,n)\,\varepsilon\,\sigma_1,
$$

where `epsilon` is **machine precision**, a measure of the smallest relative numerical differences reliably represented by the floating-point system. In applied denoising, the threshold should also reflect measurement noise and task goals, not only machine precision.


## 2. Low-rank signal plus noise

Suppose a clean data matrix has two dominant latent patterns:

$$
S=UV,
$$

with rank at most `2`. Observed data are

$$
X=S+N,
$$

where `N` is noise.

The noise may introduce many small singular values. A rank-two approximation can retain the dominant signal directions while discarding some small-scale variation.

This works best when signal singular values are clearly separated from noise. If useful information lives in weak directions, aggressive truncation can remove it.


In [ ]:
rng_noise = np.random.default_rng(7)
U_signal = rng_noise.normal(size=(40, 2))
V_signal = rng_noise.normal(size=(2, 30))
clean_signal = U_signal @ V_signal
noise_matrix = 0.65 * rng_noise.normal(size=clean_signal.shape)
observed_noisy = clean_signal + noise_matrix

U_noise, s_noise, Vt_noise = np.linalg.svd(observed_noisy, full_matrices=False)
rank2_denoised = (U_noise[:, :2] * s_noise[:2]) @ Vt_noise[:2, :]

noisy_error_to_clean = np.linalg.norm(observed_noisy-clean_signal, "fro")
denoised_error_to_clean = np.linalg.norm(rank2_denoised-clean_signal, "fro")

print("Exact rank of clean signal:", np.linalg.matrix_rank(clean_signal))
print("First ten noisy singular values:", s_noise[:10])
print("Noisy error relative to clean signal:", noisy_error_to_clean)
print("Rank-two error relative to clean signal:", denoised_error_to_clean)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, matrix, title in zip(
    axes[:3],
    [clean_signal, observed_noisy, rank2_denoised],
    ["Clean rank-two signal", "Observed signal plus noise", "Rank-two SVD reconstruction"],
):
    ax.imshow(matrix, cmap="coolwarm", aspect="auto")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
axes[3].semilogy(np.arange(1, len(s_noise)+1), s_noise, marker="o", markersize=3)
axes[3].axvline(2.5, color="red", linestyle="--", label="retain two")
axes[3].set_title("Noisy singular values")
axes[3].set_xlabel("index")
axes[3].set_ylabel("singular value")
axes[3].legend()
plt.tight_layout()
plt.show()


## 3. Why truncation can denoise

The SVD orders orthogonal matrix patterns by strength. When a small number of strong coherent patterns generate the signal and noise is spread across many directions, the leading singular components have higher signal-to-noise ratios.

Discarding small components can therefore reduce noise. However:

- structured noise can also create large singular values;
- rare but important signals can create small singular values;
- the correct rank depends on the application; and
- denoising quality should be evaluated against meaningful targets.

SVD is a mathematical filter, not an automatic truth detector.


## 4. Visual application: synthetic image compression

A grayscale image is a matrix whose entries are pixel intensities. A rank-`k` approximation stores dominant separable row-column patterns.

The following image is generated mathematically so no external image file is required. It contains smooth background structure, a circular feature, and small noise.


In [ ]:
grid_img = np.linspace(-1, 1, 80)
X_img, Y_img = np.meshgrid(grid_img, grid_img)
smooth_img = 0.45 + 0.20*np.outer(np.sin(np.pi*grid_img), np.ones_like(grid_img))
smooth_img += 0.18*np.outer(np.ones_like(grid_img), np.cos(2*np.pi*grid_img))
circle_img = 0.35 * np.exp(-8*((X_img-0.25)**2 + (Y_img+0.15)**2))
rng_img = np.random.default_rng(9)
synthetic_image = np.clip(smooth_img + circle_img + 0.04*rng_img.normal(size=X_img.shape), 0, 1)

U_img, s_img, Vt_img = np.linalg.svd(synthetic_image, full_matrices=False)
ranks_img = [2, 5, 15]
approximations_img = [(U_img[:, :k] * s_img[:k]) @ Vt_img[:k, :] for k in ranks_img]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, matrix, title in zip(
    axes,
    [synthetic_image, *approximations_img],
    ["Original 80 by 80", "Rank 2", "Rank 5", "Rank 15"],
):
    ax.imshow(matrix, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.suptitle("Increasing rank preserves more image detail", y=1.02)
plt.tight_layout()
plt.show()

for k, approximation in zip(ranks_img, approximations_img):
    relative_error = np.linalg.norm(synthetic_image-approximation, "fro") / np.linalg.norm(synthetic_image, "fro")
    storage = k*(synthetic_image.shape[0] + synthetic_image.shape[1] + 1)
    print(f"rank {k:2d}: relative error={relative_error:.4f}, SVD numbers={storage}")
print("Original pixel count:", synthetic_image.size)


## 5. Storage count for explicit SVD factors

If an approximation stores

$$
U_k\in\mathbb{R}^{m\times k},
$$

the `k` singular values, and

$$
V_k\in\mathbb{R}^{n\times k},
$$

the explicit count is

$$
mk+k+nk=k(m+n+1).
$$

If singular values are absorbed into one factor, the count becomes

$$
k(m+n).
$$

Compression also depends on metadata, data types, formats, and whether specialized image codecs outperform a generic matrix factorization.


> **Common mistakes in Section 6**
>
> - Treating every small singular value as meaningless noise.
> - Using exact equality to zero for floating-point rank decisions.
> - Selecting rank only by visual appearance.
> - Comparing storage counts without using the same numerical precision.
> - Claiming a denoised matrix is the true signal when no ground truth exists.
> - Forgetting that specialized compression methods may be more appropriate for a specific data type.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Numerical rank | Count of singular values exceeding a chosen tolerance |
| Machine precision | Floating-point scale below which relative differences are unreliable |
| Latent pattern | Hidden lower-dimensional structure generating observations |
| Noise | Unwanted or unexplained variation |
| Denoising | Reducing noise while attempting to preserve signal |
| Singular-value gap | Large separation between consecutive singular values |
| Signal-to-noise ratio | Relative strength of desired structure to noise |
| Pixel matrix | Numerical array of image intensities |


## Section 6 guided worksheet

1. A data matrix has singular values

   $$
   12,\ 3,\ 0.08,\ 0.01.
   $$

   If the tolerance is $$\tau=0.05,$$ determine its numerical rank.

2. Explain the distinction between exact rank and numerical rank.

3. A grayscale image is represented by a $$100\times 80$$ matrix. If its rank-$$10$$ SVD stores $$U_{10},$$ all ten singular values, and $$V_{10},$$ compare the explicit SVD storage count with the original count. Compute the retained and reduced percentages.

4. Why can truncating small singular values denoise data, and when can this strategy fail?


In [ ]:
reveal_markdown("Section 6 worksheet solution", r"""
1. Numerical rank counts singular values strictly larger than the tolerance. Here,

   $$
   12>0.05,\qquad 3>0.05,\qquad 0.08>0.05,
   $$

   but

   $$
   0.01<0.05.
   $$

   Therefore

   $$
   \operatorname{rank}_{0.05}(A)=3.
   $$

2. Exact rank counts mathematically nonzero singular values. Numerical rank counts singular values above a tolerance selected to reflect floating-point error, measurement noise, or the application. Thus numerical rank depends on scale and purpose, while exact rank is an algebraic property.

3. The original matrix stores

   $$
   100(80)=8{,}000
   $$

   values. Explicit rank-$$10$$ SVD storage is

   $$
   10(100+80+1)=1{,}810.
   $$

   The retained percentage is

   $$
   \frac{1{,}810}{8{,}000}(100\%)=22.625\%,
   $$

   so the reduction is

   $$
   100\%-22.625\%=77.375\%.
   $$

4. When the underlying signal is approximately low rank and noise is spread across weak directions, truncation preserves dominant structure and removes some noise. It can fail when weak directions contain meaningful rare features, when noise is strong in a leading direction, or when the data are not well described by a low-rank linear model.
""")


---

# Integrated Instructor Lab: Compress a Document-Feature Matrix

Rows represent six documents. Columns represent five transparent topic features in this order:

1. linear algebra;
2. calculus;
3. probability;
4. graphs;
5. artificial intelligence.

The matrix is

$$
X=
\begin{bmatrix}
0.9&0.1&0.1&0.1&0.8\\
0.8&0.1&0.0&0.1&0.9\\
0.1&0.9&0.1&0.0&0.7\\
0.1&0.8&0.2&0.0&0.6\\
0.1&0.1&0.9&0.2&0.7\\
0.0&0.1&0.8&0.3&0.6
\end{bmatrix}.
$$

## Lab tasks

1. Compute the reduced SVD.
2. Plot the singular values and cumulative energy.
3. Construct rank-one, rank-two, and rank-three approximations.
4. Compare errors and storage counts.
5. Explain what rank selection means for retaining topic patterns.


In [ ]:
X_lab4 = np.array([
    [0.9,0.1,0.1,0.1,0.8],
    [0.8,0.1,0.0,0.1,0.9],
    [0.1,0.9,0.1,0.0,0.7],
    [0.1,0.8,0.2,0.0,0.6],
    [0.1,0.1,0.9,0.2,0.7],
    [0.0,0.1,0.8,0.3,0.6],
])
U_lab4, s_lab4, Vt_lab4 = np.linalg.svd(X_lab4, full_matrices=False)
energy_lab4 = np.cumsum(s_lab4**2) / np.sum(s_lab4**2)

print("U shape:", U_lab4.shape)
print("singular values:", s_lab4)
print("V transpose shape:", Vt_lab4.shape)

for k in [1, 2, 3]:
    Xk = (U_lab4[:, :k] * s_lab4[:k]) @ Vt_lab4[:k, :]
    error = np.linalg.norm(X_lab4-Xk, "fro")
    relative = error/np.linalg.norm(X_lab4, "fro")
    storage = k*(X_lab4.shape[0]+X_lab4.shape[1]+1)
    print(f"rank {k}: error={error:.6f}, relative={relative:.4f}, energy={energy_lab4[k-1]:.4f}, storage={storage}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(range(1, len(s_lab4)+1), s_lab4, marker="o")
axes[0].set_xlabel("component")
axes[0].set_ylabel("singular value")
axes[0].set_title("Singular-value scree plot")
axes[1].plot(range(1, len(s_lab4)+1), energy_lab4, marker="o")
axes[1].axhline(0.9, color="red", linestyle="--", label="90% target")
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("retained components")
axes[1].set_ylabel("cumulative energy")
axes[1].set_title("Cumulative singular-value energy")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
lab_matrices = [X_lab4]
lab_titles = ["Original"]
for k in [1, 2, 3]:
    lab_matrices.append((U_lab4[:, :k] * s_lab4[:k]) @ Vt_lab4[:k, :])
    lab_titles.append(f"Rank {k}")

for ax, matrix, title in zip(axes, lab_matrices, lab_titles):
    ax.imshow(matrix, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_title(title)
    ax.set_xticks(range(5), ["LA", "Calc", "Prob", "Graph", "AI"], rotation=45)
    ax.set_yticks(range(6), [f"doc {i}" for i in range(1,7)])
plt.suptitle("Document-feature reconstruction at increasing rank", y=1.04)
plt.tight_layout()
plt.show()


## Lab interpretation

The first component captures the strongest shared pattern across the matrix, which includes the broadly high AI coordinate. Additional components distinguish the linear-algebra, calculus, and probability document groups.

Rank selection determines how many independent document-feature patterns are retained. A rank-one model is very compact but merges several topic distinctions. Higher ranks preserve more topic-specific structure at the cost of more stored values.

The original matrix stores

$$
6(5)=30
$$

numbers. Explicit rank-`k` SVD factors store

$$
k(6+5+1)=12k
$$

numbers. For this tiny matrix, rank three uses `36` numbers and therefore does not compress. Low-rank factorization is most advantageous when matrix dimensions are large and the selected rank is small.

> **Important scale lesson**
>
> A mathematically lower rank does not automatically yield storage savings for a small matrix. Always perform the count.


---

# Module 4 Wrap-Up

## Mathematical storyline

1. Every real matrix has orthonormal input and output singular directions.
2. Singular values measure nonnegative stretching magnitudes.
3. Right singular vectors are eigenvectors of `A` transpose times `A`.
4. SVD maps a unit sphere to an ellipse or ellipsoid.
5. A matrix is a sum of singular-value-weighted rank-one patterns.
6. Truncated SVD keeps the strongest patterns and is optimally accurate for its rank in two important norms.
7. Thin factors can reduce AI weight-matrix storage and computation.
8. Singular-value decay exposes redundancy and can support denoising.
9. Rank, error, storage, speed, and downstream accuracy must be evaluated together.


## Formula reference

### Singular value decomposition

$$
A=U\Sigma V^T.
$$

### Singular-vector relation

$$
A\mathbf{v}_i=\sigma_i\mathbf{u}_i.
$$

### Eigenvalue connection

$$
A^TA\mathbf{v}_i=\sigma_i^2\mathbf{v}_i.
$$

### Rank-one expansion

$$
A=\sum_{i=1}^{r}\sigma_i\mathbf{u}_i\mathbf{v}_i^T.
$$

### Truncated SVD

$$
A_k=\sum_{i=1}^{k}\sigma_i\mathbf{u}_i\mathbf{v}_i^T.
$$

### Optimal errors

$$
\lVert A-A_k\rVert_2=\sigma_{k+1},
$$

$$
\lVert A-A_k\rVert_F
=\sqrt{\sum_{i=k+1}^{r}\sigma_i^2}.
$$


## Cumulative practice

Let

$$
A=
\begin{bmatrix}
2&2\\
1&1
\end{bmatrix}.
$$

1. Explain why the rank is one.
2. Write `A` as an outer product.
3. Normalize the two outer-product vectors.
4. Find the nonzero singular value.
5. Construct a compact SVD.
6. State the error of the best rank-one approximation.


In [ ]:
reveal_markdown("Open cumulative solution", r"""
### 1. Rank

The second column equals the first column, so there is only one independent column direction. The rank is one.

### 2. Outer product

$$
A=
\begin{bmatrix}2\\1\end{bmatrix}
\begin{bmatrix}1&1\end{bmatrix}.
$$

### 3. Normalize

The first vector has norm

$$
\sqrt{2^2+1^2}=\sqrt5.
$$

The second has norm

$$
\sqrt{1^2+1^2}=\sqrt2.
$$

Thus,

$$
\mathbf{u}_1=\frac{1}{\sqrt5}
\begin{bmatrix}2\\1\end{bmatrix},
\qquad
\mathbf{v}_1=\frac{1}{\sqrt2}
\begin{bmatrix}1\\1\end{bmatrix}.
$$

### 4. Singular value

The scalar absorbed by normalization is

$$
\sigma_1=\sqrt5\sqrt2=\sqrt{10}.
$$

### 5. Compact SVD

$$
A=\mathbf{u}_1[\sqrt{10}]\mathbf{v}_1^T.
$$

The compact factor shapes are `2 by 1`, `1 by 1`, and `2 by 1`.

### 6. Rank-one approximation error

The original matrix already has rank one, so its best rank-one approximation is exact. Both spectral and Frobenius errors are zero.
""")


## Exit ticket

1. What does each SVD factor do geometrically?
2. How are singular values connected to eigenvalues?
3. Why is each SVD outer-product term rank one?
4. What makes truncated SVD optimal?
5. Why must model compression be evaluated beyond matrix approximation error?
